# Study Compare Donuts — per-donut Zernike comparison between two runs (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-06-11
**Last Modified:** 2026-06-11
**Status:** Draft
**Keywords:** AOS, FAM, Danish, donut, Zernike, comparison, version, binning, algorithm, systematics

## Description

Compare per-donut wavefront Zernikes between **two processing runs** — different
code versions, binning factors, or donut algorithms — on the **same physical
donuts**.  This notebook consolidates the former `study_danish_v0p6_vs_v1`,
`study_binning`, and `donutalgo_comparison` notebooks, which all did the same
thing.

The two runs are **side A** and **side B**, each selected by a named
`param_set` (the `output/<ps>/{donuts,visits,fits}.parquet` layout) with an
explicit-path fallback for legacy tables.  For each common visit (rotator angle
near zero so OCS and CCS ~coincide) donuts are matched per-CCD by intra-focal
centroid (KDTree within `match_tol_pix`), then compared via toggleable sections:

1. **Coverage maps** — matched-donut count per `(thx, thy)` bin (OCS + CCS).
2. **Per-visit large-|Δ|** — donuts with `|Δzk| > threshold` per visit (Z5–Z8).
3. **Density** — `Δ = zk_B − zk_A` vs `zk_A` (`hist`), or `zk_B` vs `zk_A`
   hexbin with 1:1 (`hexbin`).  Full focal plane and edge-only annulus.
4. **Difference histograms** — log-y `Δ` per Zernike.
5. **Focal-plane Δ maps** — binned-median `Δzk` in OCS and CCS.
6. **DZ-fit comparison** *(optional)* — per-visit Double-Zernike coefficients
   `{prefix}_z{j}_c{k}` overplotted between the two runs (needs fits parquets).

**Output:** PDFs + a long-format parquet of matched donut pairs in
`output/study_compare_donuts/<A>_vs_<B>/`.

Shared machinery lives in `code/compare_donuts.py`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-11 | Aaron Roodman | Initial version — merge of study_danish_v0p6_vs_v1, study_binning, donutalgo_comparison. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Data — resolve inputs & common visits](#data)
4. [Donut matching](#matching)
5. [Coverage maps](#coverage)
6. [Per-visit large-|Δ| donut counts](#largedelta)
7. [Density — full focal plane](#density)
8. [Density — edge donuts only](#edge)
9. [Difference histograms](#diffhist)
10. [Focal-plane maps — OCS](#maps-ocs)
11. [Focal-plane maps — CCS](#maps-ccs)
12. [DZ fit comparison](#dz-compare)
13. [Outputs](#output)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Comparison sides (A vs B) -----
# Each side is a processing run.  param_set selects the new
# output/<ps>/{donuts,visits,fits}.parquet layout.  param_set_X also
# serves as the side's identifier in the output path and plot labels.
param_set_A = 'fam_danish_v1_triplets_bin_2x'
param_set_B = 'fam_danish_v1_triplets_bin_1x'
label_A = param_set_A
label_B = param_set_B

# Explicit-path overrides (None = derive from param_set).  Use for legacy
# tables not in the output/<ps>/ tree.  A donut override may be a single
# path or a list of per-chunk paths (pooled).  For legacy single-file
# tables, set the matching visits_override (and fits_override if using the
# DZ-fit comparison), e.g.:
#   param_set_A      = 'danish_v0p6'
#   donut_override_A = 'output/aos_fam_danish_triplets_20260315_20260409.parquet'
#   visits_override_A = 'output/aos_fam_danish_triplets_20260315_20260409_visits.parquet'
donut_override_A = None
visits_override_A = None
fits_override_A = None
donut_override_B = None
visits_override_B = None
fits_override_B = None
output_root = 'output'

# ----- Visit filter -----
rot_max_deg = 3.0        # |rotator_angle| cut so OCS/CCS ~coincide
program_filter = None    # None = no science_program cut

# ----- Donut matching -----
# Two donuts match when their intra-focal centroids lie within
# match_tol_pix on the same CCD.
match_tol_pix = 5.0

# ----- Section toggles -----
do_coverage_maps = True
do_largedelta    = True
do_density       = True
do_edge_density  = True
do_diff_hist     = True
do_maps_ocs      = True
do_maps_ccs      = True      # auto-skipped if CCS columns are absent
do_dz_compare    = False     # needs fits parquets on both sides

# ----- Density plot styling -----
density_style = 'hist'       # 'hist' (Δ vs zk_A) or 'hexbin' (zk_B vs zk_A, 1:1)
scatter_x_plo_pct = 2.0
scatter_x_phi_pct = 98.0
scatter_y_plo_pct = 1.0
scatter_y_phi_pct = 99.0
scatter_panels_per_page = 4
scatter_n_bins_per_axis = 100

# Edge-donut annulus (corner-WFS-like radius), degrees.  Upper bound
# matches the standard AOS donut cut (1.725°).
edge_radius_min_deg = 1.62
edge_radius_max_deg = 1.725

# ----- Focal-plane maps -----
map_n_bins      = 61
map_fp_radius   = 1.8
map_plo_pct     = 2.0
map_phi_pct     = 98.0
map_panels_per_page = 4

# ----- Per-visit large-|Δ| diagnostic (OCS) -----
delta_threshold_um = 0.05
large_delta_j_list = (5, 6, 7, 8)

# ----- DZ-fit comparison -----
fit_prefix = 'z1toz6'

# ----- Output -----
output_dir = f'{output_root}/study_compare_donuts/{label_A}_vs_{label_B}'
print(f'A = {label_A}\nB = {label_B}\noutput_dir = {output_dir}')

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable

sys.path.insert(0, str(Path.cwd() / 'code'))
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.zernike_names import NOLL_NAMES
import compare_donuts as cd

setup_plotting()
print('Ready.')

<a id='data'></a>
## 3. Data — resolve inputs & common visits

Resolve each side to `(donuts, visits, fits)` paths, load the visits sidecars,
apply the rotator / program filter, and take the visits common to both runs.

In [ ]:
dA, vA, fA = cd.resolve_side(param_set=param_set_A, donut=donut_override_A,
                             visits=visits_override_A, fits=fits_override_A,
                             output_root=output_root)
dB, vB, fB = cd.resolve_side(param_set=param_set_B, donut=donut_override_B,
                             visits=visits_override_B, fits=fits_override_B,
                             output_root=output_root)
print(f'Side A donuts: {dA}\nSide B donuts: {dB}')

visits_A_full = cd.load_visits(vA)
visits_B_full = cd.load_visits(vB)
print(f'Visits: A={len(visits_A_full)}, B={len(visits_B_full)}')

mA = cd.select_visits(visits_A_full, rot_max_deg=rot_max_deg,
                      program_filter=program_filter)
mB = cd.select_visits(visits_B_full, rot_max_deg=rot_max_deg,
                      program_filter=program_filter)
visits_A, visits_B = visits_A_full[mA], visits_B_full[mB]
print(f'After rot/program filter: A kept {len(visits_A)}, B kept {len(visits_B)}')

keys_A = set(zip(np.asarray(visits_A['day_obs']).astype(int).tolist(),
                 np.asarray(visits_A['seq_num']).astype(int).tolist()))
keys_B = set(zip(np.asarray(visits_B['day_obs']).astype(int).tolist(),
                 np.asarray(visits_B['seq_num']).astype(int).tolist()))
common_visits = sorted(keys_A & keys_B)
print(f'\nCommon visits in both runs: {len(common_visits)}')
if not common_visits:
    raise RuntimeError('No common visits found — check input files / cuts.')

iZs = cd.probe_noll_indices(visits_A, dA)
print(f'Pupil Noll j (n={len(iZs)}): {iZs}')

<a id='matching'></a>
## 4. Donut matching

Stream each common visit, require matched intra/extra in both runs, and
KDTree-match donuts per CCD by intra-focal centroid.  Both OCS and CCS Zernikes
plus the extra-focal field angles are accumulated (CCS only if present).

In [ ]:
src_A = cd.VisitSource(dA)
src_B = cd.VisitSource(dB)
m = cd.accumulate_matched_donuts(src_A, src_B, common_visits,
                                 iZs=iZs, tol_pix=match_tol_pix)

print(f'\nMatched donut pairs: {m.total_matched:,}')
_denom = max(1, m.total_matched + m.total_unmatched)
print(f'Unmatched donuts:    {m.total_unmatched:,}  '
      f'(≈ {m.total_unmatched / _denom:.1%})')
if m.n_matched_per_visit:
    print(f'Per-visit match counts: median = '
          f'{int(np.median(m.n_matched_per_visit))}, '
          f'range = [{min(m.n_matched_per_visit)}, '
          f'{max(m.n_matched_per_visit)}]')
has_ccs = m.zk_a_CCS is not None
print(f'CCS columns present: {has_ccs}')

<a id='coverage'></a>
## 5. Coverage maps (sanity-check)

Number of matched donut pairs per `(thx, thy)` bin in OCS (and CCS, if present),
on a shared log color scale.  Coverage should show the FAM 9-detector
fingerprint.

In [ ]:
if do_coverage_maps and m.total_matched:
    _xb = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)
    _cO, _, _ = np.histogram2d(m.thy_OCS, m.thx_OCS, bins=[_xb, _xb])
    _vmax = _cO.max()
    panels = [('OCS', m.thx_OCS, m.thy_OCS)]
    if has_ccs:
        _cC, _, _ = np.histogram2d(m.thy_CCS, m.thx_CCS, bins=[_xb, _xb])
        _vmax = max(_vmax, _cC.max())
        panels.append(('CCS', m.thx_CCS, m.thy_CCS))
    panel = cd.make_count_panel(fp_radius=map_fp_radius, n_bins=map_n_bins,
                                vmax_shared=float(_vmax))
    cd.stream_pdf_pages(
        panels, panels_per_page=2, ncols=2, page_size=(13, 6),
        suptitle_fmt=(f'Matched donut counts per (thx, thy) bin   '
                      f'(n_pairs={m.total_matched:,}, '
                      f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/coverage_maps.pdf', plot_panel=panel)
else:
    print('skipped (do_coverage_maps=False or no matches)')

<a id='largedelta'></a>
## 6. Per-visit large-|Δ| donut counts

Per visit, how many matched donuts have `|Δzk_OCS| > delta_threshold_um` in
Z5–Z8, overlaid with the total matched-donut count.  If the spatial Δ pattern is
a clean per-donut bias, the large-|Δ| count tracks the matched count; if it is
dominated by a few visits, the pattern is a matching artefact.

In [ ]:
if do_largedelta and m.total_matched:
    _iz = {int(j): i for i, j in enumerate(iZs)}
    _dfv = pd.DataFrame({'day_obs': m.day_obs, 'seq_num': m.seq_num})
    _js = []
    for _j in large_delta_j_list:
        if int(_j) in _iz:
            _dfv[f'abs_delta_z{int(_j)}'] = np.abs(m.delta_OCS(_iz[int(_j)]))
            _js.append(int(_j))
        else:
            print(f'Z{_j} not in iZs — skipping')
    _g = _dfv.groupby(['day_obs', 'seq_num'], sort=True)
    _total = _g.size()
    _ordinal = np.arange(len(_total))
    _large = {_j: _g[f'abs_delta_z{_j}'].apply(
        lambda s, thr=delta_threshold_um: int((s > thr).sum())) for _j in _js}

    fig, ax = plt.subplots(figsize=(14, 5.5), layout='constrained')
    ax.plot(_ordinal, _total.values, '.-', color='gray', lw=1.0, ms=4,
            alpha=0.9, label=f'matched donut pairs ({int(_total.sum()):,})')
    _pal = {5: 'tab:blue', 6: 'tab:orange', 7: 'tab:green', 8: 'tab:red'}
    for _j in _js:
        ax.plot(_ordinal, _large[_j].values, '.-', lw=1.0, ms=3, alpha=0.85,
                color=_pal.get(_j),
                label=(f'Z{_j} |Δ|>{delta_threshold_um:g} μm   '
                       f'({int(_large[_j].sum()):,})'))
    ax.set_xlabel('visit ordinal (sorted by day_obs, seq_num)')
    ax.set_ylabel('donut count per visit')
    ax.set_title(f'Per-visit donut counts: total matched vs '
                 f'|Δzk_OCS|>{delta_threshold_um:g} μm   '
                 f'(B={label_B} − A={label_A}, |rot|<{rot_max_deg:g}°)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9, loc='best')
    ax.set_ylim(bottom=0)
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    with PdfPages(f'{output_dir}/largedelta_per_visit.pdf') as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    print(f'Wrote {output_dir}/largedelta_per_visit.pdf')
else:
    print('skipped (do_largedelta=False or no matches)')

<a id='density'></a>
## 7. Density — full focal plane

Per Zernike density of side B vs side A.  `density_style='hist'` plots
`Δ = zk_B − zk_A` vs `zk_A` (log color, OLS overlay); `'hexbin'` plots
`zk_B` vs `zk_A` with a 1:1 line (donutalgo style).  Uses OCS Zernikes.

In [ ]:
def _density_panels(mask=None):
    sel = slice(None) if mask is None else mask
    return [(j, m.zk_a_OCS[sel, i], m.zk_b_OCS[sel, i])
            for i, j in enumerate(iZs)]


def _make_density_panel():
    if density_style == 'hexbin':
        return cd.make_hexbin_panel(
            plo=scatter_x_plo_pct, phi=scatter_x_phi_pct,
            xlabel=f'{label_A} (OCS) [μm]',
            ylabel=f'{label_B} (OCS) [μm]')
    return cd.make_density_panel(
        x_plo=scatter_x_plo_pct, x_phi=scatter_x_phi_pct,
        y_plo=scatter_y_plo_pct, y_phi=scatter_y_phi_pct,
        n_bins=scatter_n_bins_per_axis,
        xlabel=f'{label_A} (OCS) [μm]',
        ylabel=f'{label_B} − {label_A} (OCS) [μm]')


if do_density and m.total_matched:
    cd.stream_pdf_pages(
        _density_panels(), panels_per_page=scatter_panels_per_page, ncols=2,
        page_size=(11, 9),
        suptitle_fmt=(f'{label_B} vs {label_A}  per-donut density ({density_style}, OCS)  '
                      f'(n_pairs={m.total_matched:,}, '
                      f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/density_{density_style}.pdf',
        plot_panel=_make_density_panel())
else:
    print('skipped (do_density=False or no matches)')

<a id='edge'></a>
## 8. Density — edge donuts only

Same density panels restricted to the focal-plane edge annulus
`edge_radius_min_deg ≤ r ≤ edge_radius_max_deg` (corner-WFS-like radius).

In [ ]:
if do_edge_density and m.total_matched:
    _r = np.hypot(m.thx_OCS, m.thy_OCS)
    _edge = (np.isfinite(_r) & (_r >= edge_radius_min_deg)
             & (_r <= edge_radius_max_deg))
    print(f'Edge donuts in [{edge_radius_min_deg}, {edge_radius_max_deg}]°: '
          f'{int(_edge.sum()):,} / {m.total_matched:,}')
    cd.stream_pdf_pages(
        _density_panels(_edge), panels_per_page=scatter_panels_per_page,
        ncols=2, page_size=(11, 10),
        suptitle_fmt=(f'{label_B} vs {label_A}  edge donuts ({density_style}, OCS)  '
                      f'(r∈[{edge_radius_min_deg:g},{edge_radius_max_deg:g}]°, '
                      f'n={int(_edge.sum()):,})  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/density_{density_style}_edge.pdf',
        plot_panel=_make_density_panel())
else:
    print('skipped (do_edge_density=False or no matches)')

<a id='diffhist'></a>
## 9. Difference histograms

Log-y histogram of `Δ = zk_B − zk_A` (OCS) per Zernike, with mean/σ annotated.

In [ ]:
if do_diff_hist and m.total_matched:
    cd.stream_pdf_pages(
        _density_panels(), panels_per_page=scatter_panels_per_page, ncols=2,
        page_size=(11, 9),
        suptitle_fmt=(f'Δzk = {label_B} − {label_A} (OCS)   '
                      f'(n_pairs={m.total_matched:,})  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/diff_histograms.pdf',
        plot_panel=cd.make_diff_hist_panel(
            plo=scatter_y_plo_pct, phi=scatter_y_phi_pct,
            n_bins=scatter_n_bins_per_axis))
else:
    print('skipped (do_diff_hist=False or no matches)')

<a id='maps-ocs'></a>
## 10. Focal-plane maps — OCS

Binned-median `Δzk = zk_B − zk_A` in `(thx_OCS, thy_OCS)` bins; symmetric
RdBu_r color from the 2–98 % percentile of the per-bin medians.

In [ ]:
if do_maps_ocs and m.total_matched:
    cd.stream_pdf_pages(
        [(j, m.delta_OCS(i)) for i, j in enumerate(iZs)],
        panels_per_page=map_panels_per_page, ncols=2, page_size=(11, 10),
        suptitle_fmt=(f'Focal-plane Δzk = {label_B} − {label_A} (OCS)  '
                      f'(n_pairs={m.total_matched:,}, {map_n_bins}×{map_n_bins} bins, '
                      f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/focalmaps_OCS.pdf',
        plot_panel=cd.make_map_panel(
            m.thx_OCS, m.thy_OCS, 'OCS', fp_radius=map_fp_radius,
            n_bins=map_n_bins, plo=map_plo_pct, phi=map_phi_pct,
            title_suffix=f'  —  {label_B} − {label_A}'))
else:
    print('skipped (do_maps_ocs=False or no matches)')

<a id='maps-ccs'></a>
## 11. Focal-plane maps — CCS

Same as §10 but in CCS — uses `zk_CCS` for Δ and `(thx_CCS, thy_CCS)` for the
bins, so any CCD-fixed (rather than sky-fixed) bias shows up here.

In [ ]:
if do_maps_ccs and has_ccs and m.total_matched:
    cd.stream_pdf_pages(
        [(j, m.delta_CCS(i)) for i, j in enumerate(iZs)],
        panels_per_page=map_panels_per_page, ncols=2, page_size=(11, 10),
        suptitle_fmt=(f'Focal-plane Δzk = {label_B} − {label_A} (CCS)  '
                      f'(n_pairs={m.total_matched:,}, {map_n_bins}×{map_n_bins} bins, '
                      f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
        output_pdf=f'{output_dir}/focalmaps_CCS.pdf',
        plot_panel=cd.make_map_panel(
            m.thx_CCS, m.thy_CCS, 'CCS', fp_radius=map_fp_radius,
            n_bins=map_n_bins, plo=map_plo_pct, phi=map_phi_pct,
            title_suffix=f'  —  {label_B} − {label_A}'))
elif do_maps_ccs:
    print('CCS columns absent — skipping CCS maps.')
else:
    print('skipped (do_maps_ccs=False)')

<a id='dz-compare'></a>
## 12. DZ fit comparison (optional)

For every visit common to both runs' fits parquets, overplot the focal-plane
Double-Zernike coefficients `{prefix}_z{j}_c{k}` — one page per pupil Noll `j`,
2×3 grid over focal Noll `k = 1..6`.  Side A = filled blue circles, side B =
open orange squares.  Needs `do_dz_compare=True` and fits parquets on both
sides.

In [ ]:
def _good_fits(ft):
    keep = np.ones(len(ft), dtype=bool)
    for bf in (f'{fit_prefix}_bad_fit', 'bad_fit'):
        if bf in ft.colnames:
            keep &= ~np.asarray(ft[bf]).astype(bool)
            break
    if rot_max_deg is not None and 'rotator_angle' in ft.colnames:
        rot = np.asarray(ft['rotator_angle'], dtype=float)
        keep &= np.isfinite(rot) & (np.abs(rot) <= rot_max_deg)
    if program_filter is not None and 'science_program' in ft.colnames:
        sp = np.asarray(ft['science_program']).astype(str)
        ps = ({str(x) for x in program_filter}
              if isinstance(program_filter, (list, tuple, set))
              else {str(program_filter)})
        keep &= np.array([s in ps for s in sp])
    return keep


def _dz_dividers(ax, seq):
    for i in range(1, len(seq)):
        if seq[i] != seq[i - 1]:
            ax.axvline(i - 0.5, color='black', lw=0.4, ls=':', alpha=0.45,
                       zorder=0)


def _dz_page(j, fit_A, fit_B, idxA, idxB, ordinal, dobs):
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout='constrained')
    axes = axes.ravel()
    for ki, k in enumerate(range(1, 7)):
        ax = axes[ki]
        col, ecol = f'{fit_prefix}_z{j}_c{k}', f'{fit_prefix}_z{j}_c{k}_err'
        if col not in fit_A.colnames or col not in fit_B.colnames:
            ax.set_visible(False)
            continue
        yA = np.asarray(fit_A[col], dtype=float)[idxA]
        yB = np.asarray(fit_B[col], dtype=float)[idxB]
        eA = (np.asarray(fit_A[ecol], dtype=float)[idxA]
              if ecol in fit_A.colnames else None)
        eB = (np.asarray(fit_B[ecol], dtype=float)[idxB]
              if ecol in fit_B.colnames else None)
        ax.errorbar(ordinal, yA, yerr=eA, fmt='o', color='tab:blue',
                    mfc='tab:blue', ms=4, ls='none', elinewidth=0.5,
                    capsize=0, alpha=0.85, label=label_A)
        ax.errorbar(ordinal, yB, yerr=eB, fmt='s', color='tab:orange',
                    mfc='none', ms=4, ls='none', elinewidth=0.5, capsize=0,
                    alpha=0.85, label=label_B)
        ax.axhline(0, color='gray', lw=0.5, alpha=0.5, zorder=0)
        _dz_dividers(ax, dobs)
        ax.set_xlabel('visit ordinal (chronological)', fontsize=9)
        ax.set_ylabel('coeff [μm]', fontsize=9)
        ax.set_title(f'k={k} ({NOLL_NAMES.get(k, "?")})', fontsize=10)
        ax.tick_params(labelsize=8)
        if ki == 0:
            ax.legend(fontsize=8, loc='best')
    fig.suptitle(f'Pupil Z{j} ({NOLL_NAMES.get(j, "?")})  —  '
                 f'DZ fits {label_B} vs {label_A}  (n_visits={len(ordinal)}, '
                 f'|rot|≤{rot_max_deg:g}°)', fontsize=12)
    return fig


if do_dz_compare:
    for _p in (fA, fB):
        if not Path(_p).exists():
            raise FileNotFoundError(_p)
    fit_A = QTable.read(str(fA))
    fit_B = QTable.read(str(fB))
    fit_A = fit_A[_good_fits(fit_A)]
    fit_B = fit_B[_good_fits(fit_B)]
    _kA = list(zip(np.asarray(fit_A['day_obs']).astype(int).tolist(),
                   np.asarray(fit_A['seq_num']).astype(int).tolist()))
    _kB = list(zip(np.asarray(fit_B['day_obs']).astype(int).tolist(),
                   np.asarray(fit_B['seq_num']).astype(int).tolist()))
    _iA = {k: i for i, k in enumerate(_kA)}
    _iB = {k: i for i, k in enumerate(_kB)}
    _common = sorted(set(_iA) & set(_iB))
    print(f'DZ visits: A={len(fit_A)}, B={len(fit_B)}, common={len(_common)}')
    if not _common:
        raise RuntimeError('No common (day_obs, seq_num) in the two fits parquets.')
    _idxA = np.array([_iA[k] for k in _common])
    _idxB = np.array([_iB[k] for k in _common])
    _dobs = np.array([k[0] for k in _common])
    _ord = np.arange(len(_common))
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    _n = 0
    with PdfPages(f'{output_dir}/dz_compare.pdf') as pdf:
        for j in iZs:
            if not any(f'{fit_prefix}_z{j}_c{k}' in fit_A.colnames
                       for k in range(1, 7)):
                continue
            fig = _dz_page(j, fit_A, fit_B, _idxA, _idxB, _ord, _dobs)
            pdf.savefig(fig, bbox_inches='tight')
            if _n == 0:
                plt.show()
            plt.close(fig)
            _n += 1
    print(f'Wrote {_n} DZ-compare pages to {output_dir}/dz_compare.pdf')
else:
    print('skipped (do_dz_compare=False)')

<a id='output'></a>
## 13. Outputs

Long-format parquet with one row per matched donut pair — `(day_obs, seq_num,
detector, thx/thy)` plus per-Zernike `zk_A`, `zk_B`, `delta = zk_B − zk_A` in
OCS (and CCS if present) — plus a per-Zernike Δ summary table.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
out = {
    'day_obs':     m.day_obs,
    'seq_num':     m.seq_num,
    'detector':    m.detector,
    'thx_OCS_deg': m.thx_OCS,
    'thy_OCS_deg': m.thy_OCS,
}
if has_ccs:
    out['thx_CCS_deg'] = m.thx_CCS
    out['thy_CCS_deg'] = m.thy_CCS
for i, j in enumerate(iZs):
    out[f'zk_A_OCS_z{j}'] = m.zk_a_OCS[:, i]
    out[f'zk_B_OCS_z{j}'] = m.zk_b_OCS[:, i]
    out[f'delta_OCS_z{j}'] = m.delta_OCS(i)
    if has_ccs:
        out[f'zk_A_CCS_z{j}'] = m.zk_a_CCS[:, i]
        out[f'zk_B_CCS_z{j}'] = m.zk_b_CCS[:, i]
        out[f'delta_CCS_z{j}'] = m.delta_CCS(i)
df_out = pd.DataFrame(out)
_table_path = f'{output_dir}/compare_pairs.parquet'
df_out.to_parquet(_table_path)
print(f'Saved {len(df_out):,} matched donut pairs -> {_table_path}')
print(f'Columns: {len(df_out.columns)}')

rows = []
for i, j in enumerate(iZs):
    d = m.delta_OCS(i)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        continue
    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    rows.append({'j': int(j), 'name': NOLL_NAMES.get(j, '?'),
                 'n': int(len(d)), 'mean': float(np.mean(d)),
                 'median': med, 'sigma_mad': 1.4826 * mad})
df_summary = pd.DataFrame(rows)
with pd.option_context('display.max_rows', None,
                       'display.float_format', '{:+.4f}'.format):
    display(df_summary)